# GLOSSARY PARSING and CREATION

In [5]:
import xml.etree.ElementTree as ET
import re
import json
import csv
from dataclasses import dataclass, field, asdict
from typing import Optional
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
"""
CREATED USING CLAUDE, following the IBFD procedure for their gazetteer:
IBFD International Tax Glossary — Parser
=========================================
Parses the IBFD ITG XML into structured records.

Choices:
  - Redirect entries ("See: X"): store reference only, don't resolve
  - Taxtopics: v3.0 only
  - Glossrefs in definitions: both plain text AND tagged [term] version
  - Glossref target IDs captured for graph lookups
"""

import xml.etree.ElementTree as ET
import re
import json
import csv
from dataclasses import dataclass, field, asdict
from typing import Optional

INPUT_PATH = "itg_classified.json"
THRESHOLD = 0.0
OUTPUT_PATH = "ibfd_embedding_classified.csv"


# ============================================================================
# DATA MODEL
# ============================================================================

@dataclass
class GlossaryEntry:
    """A single parsed glossary entry."""
    term: str                                        # The glossary term
    term_id: str                                     # XML id attribute
    entry_type: str                                  # "definition" | "redirect" | "multi_definition"
    definitions: list[str] = field(default_factory=list)       # Plain text definition(s)
    definitions_tagged: list[str] = field(default_factory=list) # Definitions with [glossref] markers
    glossref_targets: list[dict] = field(default_factory=list)  # [{term, target_id}] from inline refs
    see_refs: list[str] = field(default_factory=list)          # "See: X" redirect targets
    see_also_refs: list[str] = field(default_factory=list)     # "See also" related terms
    synonyms: list[str] = field(default_factory=list)          # Alternative names
    country: Optional[str] = None                              # Country if jurisdiction-specific
    taxtopics: list[dict] = field(default_factory=list)        # v3.0 taxtopics [{tc, score, label}]


# ============================================================================
# TEXT EXTRACTION HELPERS
# ============================================================================

def extract_text(element) -> str:
    """Recursively extract all text from an XML element, stripping all tags."""
    parts = []
    if element.text:
        parts.append(element.text)
    for child in element:
        parts.append(extract_text(child))
        if child.tail:
            parts.append(child.tail)
    return " ".join(parts)


def extract_text_tagged(element) -> str:
    """
    Extract text from an XML element, wrapping <glossref> content in [brackets].
    All other tags are stripped as usual.
    
    Example output: "Reduction, generally of tax or of the [tax base]."
    """
    parts = []
    if element.text:
        parts.append(element.text)
    for child in element:
        if child.tag == "glossref":
            # Wrap glossref content in brackets
            inner = extract_text(child)  # plain text inside the glossref
            parts.append(f"[{inner.strip()}]")
        else:
            # Recurse for other tags (emph, etc.)
            parts.append(extract_text_tagged(child))
        if child.tail:
            parts.append(child.tail)
    return " ".join(parts)


def extract_glossrefs(element) -> list[dict]:
    """
    Extract all <glossref> elements from an XML element.
    Returns list of {term, target_id} dicts.
    """
    refs = []
    for glossref in element.iter("glossref"):
        target_id = glossref.get("target", "")
        ref_text = clean_text(extract_text(glossref))
        if ref_text and target_id:
            refs.append({"term": ref_text, "target_id": target_id})
    return refs


def clean_text(text: str) -> str:
    """Normalize whitespace and strip."""
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_synonym(text: str) -> str:
    """Clean synonym text — strip stray brackets from synonym list formatting."""
    text = clean_text(text)
    text = re.sub(r"^\[?\s*", "", text)
    text = re.sub(r"\s*\]?$", "", text)
    return text


def extract_country(glossterm_el) -> tuple[str, Optional[str]]:
    """
    Extract the term text and country (if present) from a <glossterm> element.
    Returns (clean_term, country_or_None).
    """
    country_el = glossterm_el.find("country")
    country = None
    if country_el is not None:
        country = clean_text(extract_text(country_el)).strip("() ")

    raw_term = clean_text(extract_text(glossterm_el))

    # Remove the country portion from the term text
    if country:
        # Country appears in various forms: (France), (e.g. France), (US)
        # Remove patterns like "(France)" or "(e.g. Liechtenstein)" from term
        raw_term = re.sub(r"\(\s*(e\.g\.?\s*)?" + re.escape(country) + r"\s*\)", "", raw_term)
        raw_term = re.sub(r"\s+", " ", raw_term).strip()

    return raw_term, country


# ============================================================================
# PARSER
# ============================================================================

def parse_entry(glossentry) -> GlossaryEntry:
    """Parse a single <glossentry> element into a GlossaryEntry."""

    entry_id = glossentry.get("id", "")

    # --- Term + Country ---
    glossterm_el = glossentry.find("glossterm")
    term, country = extract_country(glossterm_el)

    # --- Synonyms ---
    synonyms = []
    synlist = glossentry.find("synonymlist")
    if synlist is not None:
        for syn in synlist.iter("synonym"):
            syn_text = clean_synonym(extract_text(syn))
            # Remove nested country text from synonym
            country_el = syn.find("country")
            if country_el is not None:
                country_text = clean_text(extract_text(country_el))
                syn_text = syn_text.replace(country_text, "").strip().strip("() ")
                syn_text = re.sub(r"\s+", " ", syn_text).strip()
            if syn_text:
                synonyms.append(syn_text)

    # --- Definitions, See refs, See-also refs ---
    definitions = []
    definitions_tagged = []
    glossref_targets = []
    see_refs = []
    see_also_refs = []

    for defn in glossentry.findall("definition"):
        # "See" redirects
        for seelist in defn.findall("seelist"):
            for see in seelist.iter("see"):
                ref_text = clean_text(extract_text(see))
                if ref_text:
                    see_refs.append(ref_text)

        # Actual definition paragraphs — plain + tagged
        for definiens in defn.findall("definiens"):
            for para in definiens.findall("para"):
                plain = clean_text(extract_text(para))
                tagged = clean_text(extract_text_tagged(para))
                if plain:
                    definitions.append(plain)
                    definitions_tagged.append(tagged)
                # Collect glossref targets from this paragraph
                glossref_targets.extend(extract_glossrefs(para))

        # "See also" references
        for salist in defn.findall("see-alsolist"):
            for sa in salist.iter("see-also"):
                ref_text = clean_text(extract_text(sa))
                if ref_text:
                    see_also_refs.append(ref_text)

    # Deduplicate glossref_targets (same ref can appear multiple times)
    seen_targets = set()
    unique_targets = []
    for ref in glossref_targets:
        key = ref["target_id"]
        if key not in seen_targets:
            seen_targets.add(key)
            unique_targets.append(ref)
    glossref_targets = unique_targets

    # --- Tax topics (v3.0 only) ---
    taxtopics = []
    for tt_block in glossentry.findall("taxtopics"):
        if tt_block.get("version") == "3.0":
            for tt in tt_block.findall("taxtopic"):
                taxtopics.append({
                    "tc": tt.get("tc", ""),
                    "score": int(tt.get("score", "0")),
                    "label": clean_text(extract_text(tt)),
                })
            break  # only one v3.0 block expected

    # If no v3.0 block found, fall back to first unversioned block
    if not taxtopics:
        for tt_block in glossentry.findall("taxtopics"):
            if not tt_block.get("version"):
                for tt in tt_block.findall("taxtopic"):
                    taxtopics.append({
                        "tc": tt.get("tc", ""),
                        "score": int(tt.get("score", "0")),
                        "label": clean_text(extract_text(tt)),
                    })
                break

    # --- Determine entry type ---
    if definitions and len(definitions) > 1:
        entry_type = "multi_definition"
    elif definitions:
        entry_type = "definition"
    elif see_refs:
        entry_type = "redirect"
    else:
        entry_type = "definition"  # edge case: entry with only taxtopics

    return GlossaryEntry(
        term=term,
        term_id=entry_id,
        entry_type=entry_type,
        definitions=definitions,
        definitions_tagged=definitions_tagged,
        glossref_targets=glossref_targets,
        see_refs=see_refs,
        see_also_refs=see_also_refs,
        synonyms=synonyms,
        country=country,
        taxtopics=taxtopics,
    )


def parse_glossary(xml_path: str) -> list[GlossaryEntry]:
    """Parse the full IBFD glossary XML."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    entries = []
    for glossentry in root.iter("glossentry"):
        entry = parse_entry(glossentry)
        entries.append(entry)

    return entries


# ============================================================================
# OUTPUT
# ============================================================================

def export_csv(entries: list[GlossaryEntry], path: str):
    """Export parsed entries to CSV."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "term", "term_id", "entry_type", "country",
            "definition_1_plain", "definition_1_tagged",
            "definition_2_plain", "definition_2_tagged",
            "glossref_count", "glossref_targets",
            "see_refs", "see_also_refs", "synonyms",
            "taxtopic_labels", "taxtopic_codes",
        ])
        for e in entries:
            writer.writerow([
                e.term,
                e.term_id,
                e.entry_type,
                e.country or "",
                e.definitions[0] if len(e.definitions) > 0 else "",
                e.definitions_tagged[0] if len(e.definitions_tagged) > 0 else "",
                e.definitions[1] if len(e.definitions) > 1 else "",
                e.definitions_tagged[1] if len(e.definitions_tagged) > 1 else "",
                len(e.glossref_targets),
                "; ".join(f"{r['term']} ({r['target_id']})" for r in e.glossref_targets),
                "; ".join(e.see_refs),
                "; ".join(e.see_also_refs),
                "; ".join(e.synonyms),
                "; ".join(t["label"] for t in e.taxtopics),
                "; ".join(t["tc"] for t in e.taxtopics),
            ])


def export_json(entries: list[GlossaryEntry], path: str):
    """Export parsed entries to JSON."""
    data = [asdict(e) for e in entries]
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)


def print_summary(entries: list[GlossaryEntry]):
    """Print parsing summary."""
    from collections import Counter

    type_counts = Counter(e.entry_type for e in entries)
    country_entries = [e for e in entries if e.country]
    synonym_entries = [e for e in entries if e.synonyms]
    topic_entries = [e for e in entries if e.taxtopics]

    total = len(entries)
    print(f"\n{'='*60}")
    print(f"IBFD GLOSSARY PARSE SUMMARY")
    print(f"{'='*60}")
    print(f"Total entries:           {total}")
    print(f"\nBy entry type:")
    for etype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
        print(f"  {etype:<25s} {count:>5d}  ({100*count/total:.1f}%)")
    print(f"\nEntries with country:    {len(country_entries)}")
    print(f"Entries with synonyms:   {len(synonym_entries)}")
    print(f"Entries with taxtopics:  {len(topic_entries)}")

    # Show a few examples of each type
    for etype in ["definition", "multi_definition", "redirect"]:
        examples = [e for e in entries if e.entry_type == etype][:3]
        if examples:
            print(f"\n--- {etype} examples ---")
            for e in examples:
                if e.definitions:
                    preview = e.definitions[0][:100] + "..."
                    print(f"  {e.term:<40s} | {preview}")
                elif e.see_refs:
                    print(f"  {e.term:<40s} | -> See: {', '.join(e.see_refs)}")
                else:
                    print(f"  {e.term:<40s} | (no content)")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == "__main__":
    import sys

    xml_path = r"itg.xml"

    print(f"Parsing {xml_path}...")
    entries = parse_glossary(xml_path)

    print_summary(entries)

    csv_path = "itg_parsed.csv"
    json_path = "itg_parsed.json"

    export_csv(entries, csv_path)
    export_json(entries, json_path)

    print(f"\nExported to:")
    print(f"  CSV:  {csv_path}")
    print(f"  JSON: {json_path}")

In [7]:
def parse_glossary_terms(xml_path): #made with the help of claude for the regex part
    tree = ET.parse(xml_path)
    root = tree.getroot()
    terms = []
    def get_text(el): #get all text from an XML element and its cildren
        parts = []
        if el.text: parts.append(el.text) #add the elements text
        for child in el:
            parts.append(get_text(child)) #extract text from child elements and any text following them
            if child.tail: parts.append(child.tail)
        return " ".join(parts) #join all text into one string
    for glossentry in root.iter("glossentry"): #go over every glosary entry
        glossterm = glossentry.find("glossterm") #find term contained within the glossary entry
        if glossterm is None: continue #skip ones that dont have a glossary term
        term = re.sub(r"\s+", " ", get_text(glossterm)).strip() #extract full text of gloss term
        country_el = glossterm.find("country") #check if the term has a country element
        if country_el is not None: #if yes, extract and clean it
            country = re.sub(r"\s+", " ", get_text(country_el)).strip().strip("() ")
            if country: #remove country from the term where it happens in parantheses
                term = re.sub(r"\(\s*(e\.g\.?\s*)?" + re.escape(country) + r"\s*\)", "", term)
                term = re.sub(r"\s+", " ", term).strip()
        if term: 
            terms.append(term) #add cleaned term to the list if not empty
    return terms

In [19]:
glossary_terms = parse_glossary_terms("itg.xml")
last_words = Counter() #see how often each word is the final word of a gloss term
for term in glossary_terms:
    words = term.lower().split()
    if words:
        last_words[words[-1]] += 1
#get the most common 30 final words
df = pd.DataFrame(last_words.most_common(30), columns=["last_word", "count"])
df["%"] = (100 * df["count"] / len(glossary_terms)).round(1) # % of terms ending with each word
print(f"total gloss terms: {len(glossary_terms)}\n")
print(df.to_string(index=False))


total gloss terms: 3763

  last_word  count   %
        tax    274 7.3
     method     74 2.0
    company     63 1.7
     income     53 1.4
transaction     38 1.0
     relief     36 1.0
  agreement     35 0.9
     system     34 0.9
      value     33 0.9
   taxation     32 0.9
  principle     31 0.8
     credit     31 0.8
       rule     25 0.7
   approach     23 0.6
 assessment     23 0.6
 adjustment     21 0.6
      basis     20 0.5
     treaty     20 0.5
    capital     19 0.5
      trust     18 0.5
       rate     18 0.5
      taxes     18 0.5
  directive     17 0.5
     ruling     16 0.4
       bond     16 0.4
       plan     16 0.4
       duty     15 0.4
      stock     15 0.4
     assets     15 0.4
    account     14 0.4


In [ ]:
#cleared output as sensitive company data
print("TAX_TYPE headwords — examples")
for word in ["tax", "duty", "levy", "tariff", "surcharge", "excise"]:
    matching_terms = []
    for term in glossary_terms: #check the last word of every term in glossary
        if term.lower().split()[-1] == word: #if matches the words we are examining,
            matching_terms.append(term) #add it 
    print(f"*{word}* ({len(matching_terms)} terms):")
    print(f"{', '.join(matching_terms[:3])}") #see 3 examples

print("\nTAX_CONCEPT headwords — examples")
for word in ["method", "principle", "rule", "approach", "doctrine", "test"]:
    matching_terms = []
    for term in glossary_terms:
        if term.lower().split()[-1] == word:
            matching_terms.append(term)
    print(f"*{word}* ({len(matching_terms)} terms):")
    print(f"{', '.join(matching_terms[:3])}")

for word in ["company", "income", "taxation", "relief", "agreement", "system"]:
    matching_terms = []
    for term in glossary_terms:
        if term.lower().split()[-1] == word:
            matching_terms.append(term)

    print(f"*{word}* ({len(matching_terms)} terms):")
    print(f"{', '.join(matching_terms[:3])}")

TAX_TYPE headwords — examples
*tax* (274 terms):
Abattoir tax, Accessions tax, Accumulated earnings tax
*duty* (15 terms):
Anti-dumping duty, Back duty, Betting duty
*levy* (7 terms):
Bearer levy, Betterment levy, Green levy
*tariff* (6 terms):
Common customs tariff, Common external tariff, Export tariff
*surcharge* (2 terms):
Import surcharge, Surcharge
*excise* (0 terms):


TAX_CONCEPT headwords — examples
*method* (74 terms):
Accounting method, Accounts method, Accrual basis method
*principle* (31 terms):
Accretion principle, Arm’s length principle, Authoritative principle
*rule* (25 terms):
Anti-dividend streaming rule, Anti-fragmentation rule, Base erosion rule
*approach* (23 terms):
Aggregate approach, Base erosion approach, Capital allocation approach
*doctrine* (5 terms):
Choice doctrine, Fiscal nullity doctrine, General utilities doctrine
*test* (6 terms):
Active income test, Benefit test, Business purpose test
*company* (63 terms):
Administrative company, Artiste company, Aux

In [ ]:
## manually selected from the examples seen above
tax_type_headwords = { "tax", "taxes", "duty", "duties", "levy", "levies", "surcharge", "surcharges", "surtax", "surtaxes", "charge", "impost", "imposts",
    "tariff", "tariffs", "cess", "toll", "tolls", "tithe", "tithes",
    "rate", "rates"}

tax_concept_headwords = {"method", "principle", "doctrine", "test", "rule","approach", "formula", "arrangement", "scheme", "mechanism",
    "technique", "strategy"}

headword_exclusions = {"tax", "taxes", "rate", "rates", "charge", "levy", "surcharge", "surtax", "backward shifting of tax", "forward shifting of tax", "shifting of tax",
    "incidence of tax", "bill, tax", "boundary, tax", "planning, tax","method", "principle", "doctrine", "test", "rule", "approach",
    "formula", "arrangement", "scheme"}

known_tax_types = {"vat", "gst", "wht", "cit", "pit", "iht", "cgt", "paye","sdlt", "sdrt", "prt", "apt", "nic", "nics",
    "irpef", "ires", "irap", "imu", "tva", "iva", "mwst", "ust", "btw","pay as you earn", "pay-as-you-earn", "stamp duty land tax",
    "value added tax", "goods and services tax"}

known_tax_concepts = {"apa", "gaar", "poem", "beps", "pe",
    "arm's length", "permanent establishment", "transfer pricing","thin capitalization", "thin capitalisation",
    "treaty override", "treaty shopping", "beneficial owner","beneficial ownership", "place of effective management",
    "substance over form", "base erosion", "profit shifting","double taxation", "economic double taxation", "juridical double taxation"}

In [ ]:
# lets classify the glossary terms for the gazetteer!
def classify_term(term):
    """this classifies terms based on layer 1 and 2 classification,
    returning tax_type, tax_concept or none"""
    lower = term.lower().strip() #case insensitive
    words = lower.split() #split into words to look
    last = words[-1] if words else "" #at last word
    # layer 1: headword checking for exclusions
    if lower not in headword_exclusions:
        if last in tax_type_headwords: return "TAX_TYPE" 
        if last in tax_concept_headwords: return "TAX_CONCEPT"
    # layer 2: lookup checking for known words/phrases
    if lower in known_tax_types: return "TAX_TYPE"
    if lower in known_tax_concepts: return "TAX_CONCEPT"
    return None

In [46]:
GLOSSARY_XML = "itg.xml"

glossary_terms = parse_glossary_terms(GLOSSARY_XML)
print(f"Parsed:{len(glossary_terms)} terms")

tax_type_terms = []
tax_concept_terms = []

for term in glossary_terms:
    label = classify_term(term)
    if label == "TAX_TYPE": tax_type_terms.append(term)
    elif label == "TAX_CONCEPT": tax_concept_terms.append(term)

tax_type_terms = sorted(set(tax_type_terms))
tax_concept_terms = sorted(set(tax_concept_terms))

print(f"TAX_TYPE:{len(tax_type_terms)}")
print(f"TAX_CONCEPT:{len(tax_concept_terms)}")
print(f"Unclassified:{len(glossary_terms) - len(tax_type_terms) - len(tax_concept_terms)}")

Parsed:3763 terms
TAX_TYPE:357
TAX_CONCEPT:208
Unclassified:3198


In [ ]:
# and now we can write them!
with open("tax_type_terms.txt", "w", encoding="utf-8") as f:
    for t in tax_type_terms:
        f.write(t + "\n")

with open("tax_concept_terms.txt", "w", encoding="utf-8") as f:
    for t in tax_concept_terms:
        f.write(t + "\n")

print(f"tax_type_terms_list.txt ({len(tax_type_terms)} terms)")
print(f"tax_concept_terms_list.txt ({len(tax_concept_terms)} terms)")

tax_type_terms_list.txt (357 terms)
tax_concept_terms_list.txt (208 terms)
